# Exploratory Data Analysis

Describe the full prepared cohort for the Any Readmission project. Each row represents one eligible inpatient encounter selected for one patient. This notebook uses the full cohort for descriptive summaries.

## 1. Setup

Load the prepared CSV using its verified schema and manifest. Keep encounter and patient identifiers for integrity checks. Display aggregate context rather than individual records.

In [ ]:
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display

working_directory = Path.cwd().resolve()
project_root = next(
    directory
    for directory in (working_directory, *working_directory.parents)
    if (directory / "src" / "paths.py").is_file() and (directory / "requirements.txt").is_file()
)
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.eda.data import load_eda_inputs

eda = load_eda_inputs(project_root=project_root)

## 2. Cohort Overview

The preparation manifest records the number of encounters and unique patients at each stage. Cleaning preserves all raw rows. Cohort selection excludes encounters with the configured death and hospice discharge codes, then keeps the smallest eligible numeric encounter ID for each patient.

In [ ]:
print(f"Prepared cohort: {len(eda.cohort):,} rows and {eda.cohort.shape[1]} columns.")
print(f"Retained encounters with unclear discharge descriptions: {eda.unclear_discharge_count:,}.")
display(eda.cohort_flow)

**Observation.** The final stage has one row per patient. The cohort also includes encounters with unclear discharge descriptions.

**Limitation.** The smallest eligible ID identifies a representative encounter. Encounter timestamps are unavailable for establishing whether it was the first admission.

## 3. Data Dictionary

Columns are listed in prepared CSV order. The period describes the information represented by the field. The original three-category outcome remains in the cohort. Field definitions draw on [Strack et al. (2014), Table 1](https://pmc.ncbi.nlm.nih.gov/articles/PMC3996476/#tab1).

In [ ]:
with pd.option_context("display.max_colwidth", None, "display.max_columns", None):
    display(eda.column_dictionary)

## 4. Source Feature Decisions

The preparation audit identifies dropped source columns, replaced diagnosis codes, and derived fields that were considered but not created. Each reason is carried over from the approved preparation step.

In [ ]:
with pd.option_context("display.max_colwidth", None, "display.max_columns", None):
    display(eda.source_feature_decisions)

## 5. Admission Code Descriptions

Admission Type and Admission Source use separate code lists. A code with an unclear description is recorded in the lookup. An unmapped code is absent from that lookup. The table shows the codes observed in the cohort.

In [ ]:
with pd.option_context("display.max_colwidth", None):
    display(eda.admission_descriptions.loc[eda.admission_descriptions["observed"]])